<a href="https://colab.research.google.com/github/karkessler/dhbw-mathe3/blob/main/notebooks/numerik/dgl_numerisch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Numerische Lösung von Differentialgleichungen

Die meisten Differentialgleichungen lassen sich nicht in geschlossener Form loesen.
Numerische Verfahren approximieren die Loesung schrittweise, ausgehend vom Anfangswert.

Wir betrachten das Anfangswertproblem $y' = -y,\ y(0) = 1$ mit der bekannten exakten
Loesung $y(t) = e^{-t}$ und vergleichen zwei Verfahren:

- **Explizites Euler-Verfahren**: der einfachste Fall, $y_{k+1} = y_k + h \cdot f(t_k, y_k)$.
  Fehlerordnung $O(h)$, und -- wichtig fuer die Praxis -- nur fuer $0 < h < 2$ stabil
  (bei diesem Testproblem).
- **Runge-Kutta-Verfahren 4. Ordnung (RK4)**: nutzt vier Hilfsauswertungen pro Schritt
  und erreicht Fehlerordnung $O(h^4)$.

Besonders instruktiv ist der Blick auf die Schrittweite beim Euler-Verfahren: Sie
entscheidet nicht nur ueber die Genauigkeit, sondern hier sogar darueber, ob die
numerische Loesung ueberhaupt gegen die richtige Loesung konvergiert.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def f(t, y):
    """Rechte Seite der DGL y' = -y."""
    return -y


def exakt(t):
    return np.exp(-t)


def euler_explizit(f, y0, t0, t_end, h):
    n = int(round((t_end - t0) / h))
    t = np.linspace(t0, t_end, n + 1)
    y = np.zeros(n + 1)
    y[0] = y0
    for k in range(n):
        y[k + 1] = y[k] + h * f(t[k], y[k])
    return t, y


def runge_kutta_4(f, y0, t0, t_end, h):
    n = int(round((t_end - t0) / h))
    t = np.linspace(t0, t_end, n + 1)
    y = np.zeros(n + 1)
    y[0] = y0
    for k in range(n):
        k1 = f(t[k], y[k])
        k2 = f(t[k] + h / 2, y[k] + h / 2 * k1)
        k3 = f(t[k] + h / 2, y[k] + h / 2 * k2)
        k4 = f(t[k] + h, y[k] + h * k3)
        y[k + 1] = y[k] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return t, y


def main():
    y0, t0, t_end = 1.0, 0.0, 5.0

    print("Anfangswertproblem y' = -y, y(0) = 1, exakte Loesung y(t) = exp(-t)\n")

    print("Explizites Euler-Verfahren bei verschiedenen Schrittweiten h:")
    print("Verstaerkungsfaktor pro Schritt: (1 - h). Stabil (dem Betrag nach)")
    print("fuer 0 < h < 2, monoton nur fuer 0 < h <= 1.\n")
    for h in [0.1, 1.9, 2.5]:
        t, y = euler_explizit(f, y0, t0, t_end, h)
        faktor = 1 - h
        if abs(faktor) < 1e-12:
            verhalten = "konstant"
        elif faktor >= 0:
            verhalten = "monoton konvergent"
        elif abs(faktor) < 1:
            verhalten = "oszillierend, aber konvergent"
        else:
            verhalten = "oszillierend und divergent"
        print(f"  h = {h:<4} -> y({t_end}) = {y[-1]: .4f}  (exakt: {exakt(t_end):.4f})  [{verhalten}]")

    # Konvergenzvergleich Euler vs. RK4
    print("\nFehler bei t = 5 in Abhaengigkeit von der Schrittweite:")
    print(f"{'h':>6} | {'Fehler Euler':>14} | {'Fehler RK4':>14}")
    hs = [0.5, 0.25, 0.125, 0.0625]
    euler_err, rk4_err = [], []
    for h in hs:
        _, y_eu = euler_explizit(f, y0, t0, t_end, h)
        _, y_rk = runge_kutta_4(f, y0, t0, t_end, h)
        eu_err = abs(y_eu[-1] - exakt(t_end))
        rk_err = abs(y_rk[-1] - exakt(t_end))
        euler_err.append(eu_err)
        rk4_err.append(rk_err)
        print(f"{h:>6} | {eu_err:>14.2e} | {rk_err:>14.2e}")

    # Plot 1: Loesungsverlauf bei h=0.1 vs h=1.9 (Stabilitaet)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    t_fein = np.linspace(t0, t_end, 300)
    axes[0].plot(t_fein, exakt(t_fein), "k--", label="exakte Loesung")
    for h, style in [(0.1, "b.-"), (1.9, "r.-")]:
        t, y = euler_explizit(f, y0, t0, t_end, h)
        axes[0].plot(t, y, style, label=f"Euler, h={h}", markersize=4)
    axes[0].set_xlabel("t")
    axes[0].set_ylabel("y(t)")
    axes[0].set_title("Explizites Euler-Verfahren: Schrittweite und Stabilitaet")
    axes[0].legend()

    axes[1].loglog(hs, euler_err, "o-", label="Euler (Ordnung 1)")
    axes[1].loglog(hs, rk4_err, "s-", label="Runge-Kutta 4 (Ordnung 4)")
    axes[1].set_xlabel("Schrittweite h (log-Skala)")
    axes[1].set_ylabel("Fehler bei t=5 (log-Skala)")
    axes[1].set_title("Konvergenzordnung: Euler vs. RK4")
    axes[1].legend()
    axes[1].grid(True, which="both", alpha=0.3)

    fig.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()